# 02 - Logistic Regression on TF-IDF

5-fold GroupKFold. Vectorizer combines word n-grams `(1,3)` and character n-grams `(3,5)` via horizontal stacking. Saves OOF and test predictions to `predictions/`.

## Setup

In [1]:
import subprocess, sys
for pkg in ['wandb', 'pyarrow']:
    try: __import__(pkg.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

In [2]:
import os
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/input')
if IS_KAGGLE:
    DATA_DIR = Path('/kaggle/input/competitions/smart-mcq-solver-challenge')
    OUTPUT_DIR = Path('/kaggle/working')
else:
    ROOT = Path.cwd()
    if ROOT.name == 'notebooks': ROOT = ROOT.parent
    DATA_DIR = ROOT / 'data'
    OUTPUT_DIR = ROOT

PREDICTIONS_DIR = OUTPUT_DIR / 'predictions'
FEATURES_DIR = OUTPUT_DIR / 'features'
FIGURES_DIR = OUTPUT_DIR / 'figures'
for p in [PREDICTIONS_DIR, FEATURES_DIR, FIGURES_DIR]:
    p.mkdir(parents=True, exist_ok=True)
print(f'DATA_DIR={DATA_DIR}')

DATA_DIR=c:\Users\Krishna\Downloads\dl-genai-t22026-23f2001849\data


In [3]:
import wandb
WANDB_ENTITY = '23f2001849-dl-genai-project'
WANDB_PROJECT = '23f2001849-t22026'
USE_WANDB = False
try:
    if IS_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret('WANDB_API_KEY')
    else:
        key = os.environ.get('WANDB_API_KEY', '').strip()
    if key:
        wandb.login(key=key, relogin=True)
        USE_WANDB = True
except Exception as e:
    print(f'W&B setup failed: {e}')
print(f'W&B enabled: {USE_WANDB}')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\Krishna\_netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


W&B enabled: True


In [4]:
import json, time
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss

SEED = 42
OPTIONS = ['A', 'B', 'C', 'D', 'E']
NUM_CHOICES = 5
FOLDS = 5

## Load features

In [5]:
train_long = pd.read_parquet(FEATURES_DIR / 'train_long.parquet')
test_long  = pd.read_parquet(FEATURES_DIR / 'test_long.parquet')
train_meta = pd.read_parquet(FEATURES_DIR / 'train_meta.parquet')
print(f'train_long: {train_long.shape}, test_long: {test_long.shape}')

train_long: (10000, 24), test_long: (2500, 22)


## MAP@3 helpers

In [6]:
def map3_score(pred, true):
    s = 0.0
    for p, a in zip(pred, true):
        for k, L in enumerate(p[:3]):
            if L == a:
                s += 1.0 / (k + 1); break
    return s / len(true)

def scores_to_top3(grid):
    order = np.argsort(-grid, axis=1)
    return [[OPTIONS[i] for i in row[:3]] for row in order]

def long_to_grid(long_df, scores, ids_order):
    id_to_row = {q: i for i, q in enumerate(ids_order)}
    grid = np.zeros((len(ids_order), NUM_CHOICES))
    for i in range(len(long_df)):
        r = long_df.iloc[i]
        grid[id_to_row[r['id']], OPTIONS.index(r['letter'])] = scores[i]
    return grid

## TF-IDF: word (1,3) + char (3,5)

Word n-grams capture phrase-level content matches between prompt and option. Character n-grams cover morphology, misspellings, and cross-word substrings that word tokenization would split apart. The two matrices are horizontally stacked so the linear model sees both feature sets.

In [7]:
tr_texts = [f'{p} [SEP] {o}' for p, o in zip(train_long['prompt'].astype(str),
                                                train_long['option'].astype(str))]
te_texts = [f'{p} [SEP] {o}' for p, o in zip(test_long['prompt'].astype(str),
                                                test_long['option'].astype(str))]

In [8]:
word_vec = TfidfVectorizer(ngram_range=(1, 3), min_df=2, max_features=50_000,
                            sublinear_tf=True, strip_accents='unicode')
char_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2,
                            max_features=50_000, sublinear_tf=True, strip_accents='unicode')
word_vec.fit(tr_texts + te_texts)
char_vec.fit(tr_texts + te_texts)
print(f'word vocab: {len(word_vec.vocabulary_)}')
print(f'char vocab: {len(char_vec.vocabulary_)}')

word vocab: 28400
char vocab: 23901


In [9]:
Xw_tr = word_vec.transform(tr_texts)
Xc_tr = char_vec.transform(tr_texts)
Xw_te = word_vec.transform(te_texts)
Xc_te = char_vec.transform(te_texts)

X_tr = hstack([Xw_tr, Xc_tr]).tocsr()
X_te = hstack([Xw_te, Xc_te]).tocsr()
y = train_long['is_correct'].astype(int).values
folds = train_long['fold'].values
print(f'X_tr: {X_tr.shape}, X_te: {X_te.shape}')

X_tr: (10000, 52301), X_te: (2500, 52301)


## 5-fold training

Per-fold binary metrics use the standard 0.5 probability threshold. Question-level `top1` and `MAP@3` are the headline metrics; the binary values are reported for completeness.

In [10]:
if USE_WANDB:
    wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name='lr', reinit=True,
               tags=['model:lr', 'features:tfidf_word+char'],
               config={'model': 'logistic_regression', 'C': 1.0, 'solver': 'liblinear',
                       'word_ngrams': '1-3', 'char_ngrams': '3-5', 'folds': FOLDS})
    wandb.define_metric('fold_idx')
    for m in ['train_loss', 'val_loss', 'train_acc', 'val_acc']:
        wandb.define_metric(f'{m}/*', step_metric='fold_idx')

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


In [11]:
true_letters = train_meta['answer'].tolist()
true_ids_order = train_meta['id'].tolist()

oof_lr = np.zeros(len(train_long))
fold_rows = []
t0 = time.time()

for f in range(FOLDS):
    tr_m = folds != f
    va_m = folds == f
    clf = LogisticRegression(C=1.0, max_iter=1000, solver='liblinear', random_state=SEED)
    clf.fit(X_tr[tr_m], y[tr_m])
    val_prob = clf.predict_proba(X_tr[va_m])[:, 1]
    tr_prob = clf.predict_proba(X_tr[tr_m])[:, 1]
    oof_lr[va_m] = val_prob

    row = {
        'fold': f,
        'train_loss': log_loss(y[tr_m], tr_prob),
        'val_loss':   log_loss(y[va_m], val_prob),
        'train_acc':  accuracy_score(y[tr_m], (tr_prob > 0.5).astype(int)),
        'val_acc':    accuracy_score(y[va_m], (val_prob > 0.5).astype(int)),
    }
    fold_rows.append(row)
    print(f'  fold {f}: train_loss {row["train_loss"]:.4f}  val_loss {row["val_loss"]:.4f}')
    if USE_WANDB:
        wandb.log({
            'fold_idx': f,
            f'train_loss/fold_{f}': row['train_loss'],
            f'val_loss/fold_{f}':   row['val_loss'],
            f'train_acc/fold_{f}':  row['train_acc'],
            f'val_acc/fold_{f}':    row['val_acc'],
        })

elapsed = time.time() - t0
print(f'training time: {elapsed:.1f}s')

  fold 0: train_loss 0.2297  val_loss 0.5033
  fold 1: train_loss 0.2411  val_loss 0.4722
  fold 2: train_loss 0.2334  val_loss 0.4880
  fold 3: train_loss 0.2289  val_loss 0.5152
  fold 4: train_loss 0.2365  val_loss 0.4914
training time: 5.2s


### Reading the metrics

**MAP@3** and **top1_acc** are the honest headline metrics for a 5-way MCQ.

**val_acc, precision, recall, F1** come from a binary `>0.5` threshold on per-option probabilities. The class distribution is 1 correct : 4 wrong per question, so the majority-class baseline is 80% accuracy. For softmax-normalized transformer probabilities the per-option mean is `1/5 = 0.2`, so almost nothing crosses 0.5 and `recall` collapses to near zero while `precision` becomes noisy. These numbers are reported for completeness — they are not diagnostic of model quality on this task.

## OOF metrics

In [12]:
oof_pred = (oof_lr > 0.5).astype(int)
grid_lr = long_to_grid(train_long, oof_lr, true_ids_order)
top3 = scores_to_top3(grid_lr)

final = {
    'accuracy':  accuracy_score(y, oof_pred),
    'precision': precision_score(y, oof_pred, zero_division=0),
    'recall':    recall_score(y, oof_pred, zero_division=0),
    'f1':        f1_score(y, oof_pred, zero_division=0),
    'log_loss':  log_loss(y, oof_lr.clip(1e-7, 1 - 1e-7)),
    'top1_acc':  float(np.mean([t[0] == a for t, a in zip(top3, true_letters)])),
    'map3':      map3_score(top3, true_letters),
    'training_time_sec': elapsed,
}
print('LR OOF metrics:')
for k, v in final.items():
    print(f'  {k}: {v:.4f}')

if USE_WANDB:
    for k, v in final.items():
        wandb.run.summary[k] = v
    wandb.log({'fold_summary': wandb.Table(dataframe=pd.DataFrame(fold_rows))})
    wandb.finish()

LR OOF metrics:
  accuracy: 0.8030
  precision: 0.7586
  recall: 0.0220
  f1: 0.0428
  log_loss: 0.4940
  top1_acc: 0.3525
  map3: 0.5132
  training_time_sec: 5.2230


fold_idx,▁▃▅▆█
train_acc/fold_0,▁
train_acc/fold_1,▁
train_acc/fold_2,▁
train_acc/fold_3,▁
train_acc/fold_4,▁
train_loss/fold_0,▁
train_loss/fold_1,▁
train_loss/fold_2,▁
train_loss/fold_3,▁
+11,...


## Full-train and test predictions

In [13]:
clf_full = LogisticRegression(C=1.0, max_iter=1000, solver='liblinear', random_state=SEED)
clf_full.fit(X_tr, y)
test_pred = clf_full.predict_proba(X_te)[:, 1]

np.save(PREDICTIONS_DIR / 'oof_lr.npy', oof_lr)
np.save(PREDICTIONS_DIR / 'test_lr.npy', test_pred)
with open(PREDICTIONS_DIR / 'lr_metrics.json', 'w') as fh:
    json.dump(final, fh, indent=2)

print(f'saved oof_lr.npy shape={oof_lr.shape}')
print(f'saved test_lr.npy shape={test_pred.shape}')

saved oof_lr.npy shape=(10000,)
saved test_lr.npy shape=(2500,)
